# 05 - Base vs fine-tuned: prove it, do not claim it

Run the **same** test set through the base model and the served adapter. The
four numbers - schema validity, exact match, latency, cost - are what you put
on the table to defend 'migrated to a self-hosted small model and cut cost'.

In [ ]:
import sys; sys.path.insert(0, '../src')
from qlora_lab import schema, synth, dataset, predict, evaluate, train, serve, agent

In [ ]:
test = dataset.read_jsonl('../data/test.jsonl')
base_client = serve.openai_client(); BASE = 'unsloth/Qwen3-8B-bnb-4bit'
tuned_client = serve.openai_client(); TUNED = 'ticket'  # adapter name in vLLM
base_preds = [predict.extract(base_client, BASE, e['message']) for e in test]
tuned_preds = [predict.extract(tuned_client, TUNED, e['message']) for e in test]

In [ ]:
# self-hosted $/token is roughly the same for base and tuned (same GPU), so the
# cost win here is mostly fewer retries + shorter, valid outputs. The big cost
# story vs a *frontier* baseline comes from notebook 01.
rb = evaluate.evaluate(base_preds, test, in_price=0.05e-6, out_price=0.20e-6)
rt = evaluate.evaluate(tuned_preds, test, in_price=0.05e-6, out_price=0.20e-6)
print('BASE :', rb.summary())
print('TUNED:', rt.summary())
print(); print(evaluate.compare(rb, rt))

Save this table. If tuned does not clear base by 10%+ on schema validity, the
fine-tune is the problem, not the model - revisit data and masking in 02.